In [48]:
# checking system info
import sys

# for running gradient descent
import numpy as np
import torch
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

# for bringing the data in from R
import pandas as pd


verbose = False

In [49]:
print(sys.executable)

/Users/jseid1/venv311/bin/python


In [50]:

def learnQ(targets, covariates,embedding_dim,verbose):
    # unpacking inputs
    covariate_matrices = covariates
    target_vectors = targets

    # rows (num outcomes)
    Y_1 = covariate_matrices[0]
    m = Y_1.shape[0] 
    num_donors = Y_1.shape[1] 
    # Embedding dimension
    D = embedding_dim

    # Q is what we're optimizing - requires_grad=True tracks gradients
    torch.manual_seed(215)
    Q = torch.randn(num_donors, D, dtype=torch.float64, requires_grad=True)

    # --- Define the inner QP once (structure never changes) ---
    w_var = cp.Variable(D)
    # Create a parameter for each target vector
    YQ_params = [cp.Parameter((m, D)) for _ in range(len(target_vectors))]
    discrepancy = [cp.sum_squares(d.numpy() - YQ_param @ w_var) for YQ_param, d in zip(YQ_params, target_vectors)]
    # I believe this is where I'll add in the many many target and covariate matrices
    constraints = [cp.sum(w_var) == 1, w_var >= 0]
    objective = cp.Minimize(sum(discrepancy))
    prob = cp.Problem(objective, constraints)
    layer = CvxpyLayer(prob, parameters=YQ_params, variables=[w_var])

    # --- Outer optimization loop ---
    optimizer = torch.optim.Adam([Q], lr=0.01)
    l2_norm = sum

    for step in range(2000):
        optimizer.zero_grad()

        # transform the covariates using Q
        YQ_list = [Y @ Q for Y in covariate_matrices]
        # solve for w given the matrix Q
        # * unpacks the list
        w_sol, = layer(*YQ_list) 

        # use l2 norm to regularize Q
        lambda_l2_Q = 0.01 
        lambda_l2_w = 1
        l2_Q = torch.sum(Q**2)
        l2_w = torch.sum(w_sol**2)

        # loss using the optimal w for this Q
        loss = sum(torch.sum((d - YQ @ w_sol)**2) for d, YQ in zip(target_vectors, YQ_list)) + (lambda_l2_Q * l2_Q) + (lambda_l2_w * l2_w)

        # this is where Q is updated
        loss.backward()                 
        optimizer.step()
        
        if step % 200 == 0:
            print(f"Step {step:4d} | Loss: {loss.item():.8f}")

    # --- Results --- #
    if (verbose == True):
        print(f"\nFinal Loss: {loss.item():.8f}")
        print(f"Final w:    {w_sol.detach().numpy().round(4)}")
        print(f"AQ @ w:     {(AQ @ w_sol).detach().numpy().round(4)}")
        print(f"Target d:   {d.numpy()}")
        print(f"Final Q:\n {Q.detach().numpy().round(4)}")
    
    Q_final = Q.detach().numpy()
    w_final = w_sol.detach().numpy()
    return Q_final, w_final
    

## Toy Example

### Toy data set (no underlying patterns)

Notice that the number of time points is greater than the number of units. What if I change this?

In [51]:

# Data
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0], [3.0, 5.0]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0], [4.0, 6.0]], dtype=torch.float64)

d_3 = torch.tensor([4.0, 4.0], dtype=torch.float64)
Y_3 = torch.tensor([[1.0, 3.0], [2.0, 4.0]], dtype=torch.float64)

target_vectors = [d_1, d_2, d_3]
covariate_matrices = [Y_1,Y_2,Y_3]


In [52]:
print(target_vectors[0].shape)   # should be (2,)
print(covariate_matrices[0].shape)  # should be (2, 2)

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 10, False)


print(d_1)
print(Y_1@Q_final@w_final)

print(d_2)
print(Y_2@Q_final@w_final)

print(d_3)
print(Y_3@Q_final@w_final)



torch.Size([2])
torch.Size([2, 2])
Step    0 | Loss: 49.11450626
Step  200 | Loss: 17.00118149
Step  400 | Loss: 15.89905812
Step  600 | Loss: 15.89571876
Step  800 | Loss: 15.64368608
Step 1000 | Loss: 15.60265322
Step 1200 | Loss: 15.60092724
Step 1400 | Loss: 15.59762699
Step 1600 | Loss: 15.59571574
Step 1800 | Loss: 15.59392211
tensor([6., 2.], dtype=torch.float64)
tensor([4.3946, 3.5984], dtype=torch.float64)
tensor([5., 3.], dtype=torch.float64)
tensor([2.1973, 2.8021], dtype=torch.float64)
tensor([4., 4.], dtype=torch.float64)
tensor([5.1909, 4.3946], dtype=torch.float64)


Making the number of donors greater than the number of time points.
This has the effect of greatly decreasing the loss.
And it is also pretty clear that the loss doesn't depend too strongly on the embedding dimension.

In [53]:
# Data
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0, 1, 9], [3.0, 5.0, 10, 11]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0, 3, 12], [4.0, 6.0, 5, 7]], dtype=torch.float64)

target_vectors = [d_1, d_2]
covariate_matrices = [Y_1,Y_2]

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 5, False)


print(d_1)
print(Y_1@Q_final@w_final)

print(d_2)
print(Y_2@Q_final@w_final)



Step    0 | Loss: 72.60129912
Step  200 | Loss: 1.08127973
Step  400 | Loss: 0.82281019
Step  600 | Loss: 0.67314261
Step  800 | Loss: 0.57412767
Step 1000 | Loss: 0.52426619
Step 1200 | Loss: 0.50280760
Step 1400 | Loss: 0.49256146
Step 1600 | Loss: 0.48647478
Step 1800 | Loss: 0.48221224
tensor([6., 2.], dtype=torch.float64)
tensor([5.9169, 1.9300], dtype=torch.float64)
tensor([5., 3.], dtype=torch.float64)
tensor([5.0676, 3.0900], dtype=torch.float64)


# Synthetic Data

- step 1: split into train and test
- step 2: normalize
    - Use the mean and STD computed only from the train data (pre-treatment) - not the test data (post treatment!)
    - apply the standardization to both train and test data
- step 3: figure out how to implement the method on multiple time points.

In [ ]:
# read in the data
data = pd.read_csv('outcomes.csv')
metadata = pd.read_csv("metadata.csv")

# convert to numpy arrays
metadata = metadata.to_numpy()
data = data.to_numpy()

# unpack the metadata
n_units, n_timepoints, n_outcomes = metadata[:,1]

# reshape the data
Y = data.reshape(n_timepoints, n_outcomes, n_units)
print("data, reshaped: \n",Y)

test = Y[0]
train = Y[1:len(Y)]

if (verbose):
    print("Test data: \n", test)
    print("Train data: \n", train)


data:
 [[ 0.12644093  0.16067742  0.19094767]
 [ 0.01078857 -0.02627012  0.21856263]
 [ 0.05902672 -0.01832678  0.2748492 ]
 [ 0.06589244 -0.02064021  0.27310725]
 [ 0.00722449 -0.18171898  0.37436272]
 [ 0.12144852 -0.0145581   0.32810407]
 [-0.04293465 -0.34346811  0.47551931]
 [ 0.18135694 -0.00412364  0.38745324]
 [-0.08153693 -0.49366037  0.58823278]
 [ 0.24902231  0.01406777  0.45455936]
 [-0.15013707 -0.67385049  0.67094838]
 [ 0.32048251  0.03605401  0.52546031]
 [-0.20490231 -0.84020571  0.76749888]
 [ 0.37757767  0.04367521  0.58199622]
 [-0.24708736 -0.99398074  0.87662957]
 [ 0.43575196  0.05237554  0.63961126]
 [-0.29982146 -1.15830483  0.97521121]
 [ 0.49792601  0.06507563  0.70122606]
 [-0.36423422 -1.33430756  1.0621142 ]
 [ 0.54559528  0.06327094  0.74833608]]
data, reshaped: 
 [[[ 0.12644093  0.16067742  0.19094767]
  [ 0.01078857 -0.02627012  0.21856263]]

 [[ 0.05902672 -0.01832678  0.2748492 ]
  [ 0.06589244 -0.02064021  0.27310725]]

 [[ 0.00722449 -0.18171898  0.

# These aren't actually organized right! Need to put into Y_{F,t} format

In [55]:
print(train)
print(train[1:2])

[[[ 0.47551931  0.18135694]
  [-0.00412364  0.38745324]
  [-0.08153693 -0.49366037]
  [ 0.58823278  0.24902231]
  [ 0.01406777  0.45455936]
  [-0.15013707 -0.67385049]
  [ 0.67094838  0.32048251]
  [ 0.03605401  0.52546031]
  [-0.20490231 -0.84020571]
  [ 0.76749888  0.37757767]]

 [[ 0.04367521  0.58199622]
  [-0.24708736 -0.99398074]
  [ 0.87662957  0.43575196]
  [ 0.05237554  0.63961126]
  [-0.29982146 -1.15830483]
  [ 0.97521121  0.49792601]
  [ 0.06507563  0.70122606]
  [-0.36423422 -1.33430756]
  [ 1.0621142   0.54559528]
  [ 0.06327094  0.74833608]]]
[[[ 0.04367521  0.58199622]
  [-0.24708736 -0.99398074]
  [ 0.87662957  0.43575196]
  [ 0.05237554  0.63961126]
  [-0.29982146 -1.15830483]
  [ 0.97521121  0.49792601]
  [ 0.06507563  0.70122606]
  [-0.36423422 -1.33430756]
  [ 1.0621142   0.54559528]
  [ 0.06327094  0.74833608]]]
